# RAG (Retrieval-Augmented Generation) — Teaching Notebook

RAG lets a language model answer questions about documents it has never seen
during training. The model doesn't need to be retrained — relevant passages are
**retrieved at query time** and injected into the prompt.

This solves two core problems with "raw" LLMs:
1. They don't know about private / recent / specialized documents.
2. They sometimes hallucinate — confidently stating wrong facts.

## The Pipeline

| Phase | Steps | When |
|---|---|---|
| **Indexing** | Documents → Chunks → Embeddings | Once, offline |
| **Retrieval** | Query → Embed → Cosine Similarity → Top-K | Per query |
| **Generation** | Top-K chunks + Query → LLM → Answer | Per query |

Run cells from top to bottom — each stage builds on the previous one.

In [ ]:
%pip install -q sentence-transformers numpy litellm

In [ ]:
import os
import textwrap

import litellm
import numpy as np
from sentence_transformers import SentenceTransformer

# ── Configuration ── tweak these to explore how they affect RAG behaviour ──

EMBEDDING_MODEL = "all-MiniLM-L6-v2"
# Lightweight, 384-dim, runs on CPU. Swap to "all-mpnet-base-v2" for better quality.

CHUNK_SIZE    = 150   # maximum words per chunk
CHUNK_OVERLAP = 30    # words shared between consecutive chunks
TOP_K         = 3     # how many chunks to pass to the LLM

# LiteLLM model string format: "provider/model-name"
# Examples: "openai/gpt-4o-mini"  "anthropic/claude-haiku-4-5"  "ollama/llama3"
LLM_MODEL    = os.getenv("LLM_MODEL", "openai/gpt-4o-mini")
LLM_API_BASE = os.getenv("LLM_API_BASE", "")  # leave empty for cloud providers

print(f"LLM: {LLM_MODEL!r}  |  api_base: {LLM_API_BASE or '(provider default)'}")

## API Key Setup

LiteLLM reads provider-specific environment variables automatically.
Set the one that matches your chosen `LLM_MODEL`:

| Provider | Env var | Example model |
|---|---|---|
| OpenAI | `OPENAI_API_KEY` | `openai/gpt-4o-mini` |
| Anthropic | `ANTHROPIC_API_KEY` | `anthropic/claude-haiku-4-5` |
| Ollama (local) | *(none)* — set `LLM_API_BASE=http://localhost:11434` | `ollama/llama3` |

Enter the key for your provider below, or skip if already set in the environment.

In [ ]:
import getpass

# Set the env var that matches your provider (see table above).
# For Ollama or other keyless local backends, skip this cell.
if LLM_MODEL.startswith("openai/") and not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OPENAI_API_KEY: ")
elif LLM_MODEL.startswith("anthropic/") and not os.getenv("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("ANTHROPIC_API_KEY: ")

---
## Knowledge Base

In a real system these would be loaded from files, a database, or a web crawler.
They are embedded here to keep the demo self-contained — five passages covering
key AI/ML concepts.

In [ ]:
DOCUMENTS = [

    # ── Document 1 ──────────────────────────────────────────────────────────
    """
    The Transformer Architecture

    The Transformer, introduced in "Attention Is All You Need" (Vaswani et al.,
    2017), is the architecture behind every modern large language model.  Unlike
    earlier RNNs it processes all tokens in parallel, making it much faster to
    train on GPUs.

    The key innovation is self-attention: every token can directly attend to
    every other token in the same sequence.  For example, in "The animal didn't
    cross the street because it was tired", attention lets the model figure out
    that "it" refers to "animal", not "street".

    Each attention head learns three weight matrices — Query (Q), Key (K), and
    Value (V) — and computes:

        Attention(Q, K, V) = softmax( Q·Kᵀ / √d_k ) · V

    Dividing by √d_k prevents the dot products from growing too large in high
    dimensions, which would push the softmax into a region with tiny gradients.

    A Transformer layer has two sub-layers:
      • Multi-head self-attention  — captures relationships between tokens.
      • Feed-forward network       — applied independently to each position.

    Both sub-layers use residual connections and layer normalisation, which
    stabilise training of very deep stacks (hundreds of layers in GPT-4 etc.).
    """,

    # ── Document 2 ──────────────────────────────────────────────────────────
    """
    Retrieval-Augmented Generation (RAG)

    RAG is a technique that combines information retrieval with text generation.
    It was popularised by Lewis et al. (2020) at Facebook AI Research.

    The problem RAG solves: LLMs are trained on a fixed snapshot of data.
    They cannot answer questions about private documents or recent events, and
    they sometimes hallucinate plausible-sounding but wrong facts.

    RAG fixes this by retrieving relevant passages from a document store at
    query time and injecting them into the prompt.  The model is instructed to
    base its answer on those passages, which anchors it in real text.

    Classic RAG pipeline:
      1. Offline — embed all documents, store vectors in a vector database.
      2. Online  — embed the query, retrieve top-K similar chunks, append to prompt.
      3. The LLM generates an answer grounded in the retrieved context.

    Key design trade-offs:
      • Chunk size:   smaller → more precise retrieval; larger → more context per hit.
      • Top-K:        more chunks → richer context but longer (costlier) prompts.
      • Embedding model: bigger → better quality but slower indexing.
    """,

    # ── Document 3 ──────────────────────────────────────────────────────────
    """
    Embeddings and Vector Similarity

    An embedding is a dense numeric vector that represents meaning.  Two
    sentences that mean similar things will have vectors pointing in similar
    directions in the high-dimensional embedding space.

    Cosine similarity is the most common distance metric for retrieval:

        similarity(A, B) = (A · B) / (‖A‖ · ‖B‖)

    It ranges from -1 (opposite meaning) to +1 (identical meaning) and is
    independent of vector magnitude — helpful because sentence lengths vary.

    Embedding models are trained with contrastive learning: pairs of similar
    sentences are pushed close together; dissimilar sentences are pushed apart.

    Popular embedding models:
      • all-MiniLM-L6-v2       — 384 dims, fast, good quality-to-speed ratio.
      • all-mpnet-base-v2      — 768 dims, slower but more accurate.
      • text-embedding-3-small — OpenAI, strong commercial option.
      • nomic-embed-text       — open weights, competitive with commercial models.

    Vector databases (ChromaDB, FAISS, Qdrant, Pinecone) store embeddings and
    support approximate nearest-neighbour (ANN) search at scale.  For small
    corpora a plain NumPy dot-product is sufficient, as in this example.
    """,

    # ── Document 4 ──────────────────────────────────────────────────────────
    """
    Prompt Engineering

    Prompt engineering is the art of crafting the text you send to an LLM to
    reliably elicit the response you want — without changing the model weights.

    Core techniques:

    Zero-shot prompting — ask the model directly, no examples:
        "Summarise the following paragraph: …"

    Few-shot prompting — provide 2–5 input/output examples before the real
    query.  This dramatically improves performance on tasks the model has not
    been fine-tuned for.

    Chain-of-thought (CoT) — ask the model to "think step by step".  Adding
    this phrase can raise accuracy on multi-step reasoning by 10–40 pp.

    System prompts — many APIs accept a system message that sets the model's
    role, tone, and constraints before any user turn.  Example:
        "You are a helpful assistant.  Answer ONLY from the provided context.
         If the context is insufficient, say 'I don't know'."

    RAG-specific tips:
      • Tell the model explicitly where its context comes from.
      • Instruct it to admit when context is insufficient rather than guess.
      • Keep context concise — padding with irrelevant text hurts quality.
    """,

    # ── Document 5 ──────────────────────────────────────────────────────────
    """
    Neural Networks — A Refresher

    A neural network is a stack of layers.  Each layer applies a linear
    transformation (matrix multiply + bias) followed by a non-linear activation.

    Common activation functions:
      • ReLU(x)    = max(0, x)    — simple, avoids vanishing gradients, most used.
      • GELU(x)    ≈ x·Φ(x)      — smoother variant used in Transformers.
      • Sigmoid(x) = 1/(1+e⁻ˣ)  — squashes output to (0,1), used for binary tasks.

    Training uses gradient descent:
      1. Forward pass  — compute predictions.
      2. Loss          — measure how wrong the predictions are (e.g. cross-entropy).
      3. Backward pass — compute gradient of the loss w.r.t. every weight.
      4. Update        — move each weight a small step against its gradient.

    Key hyperparameters:
      • Learning rate — step size per update; too large → diverges, too small → slow.
      • Batch size    — samples per gradient step; affects stability and speed.
      • Epochs        — full passes through the training data.

    Modern LLMs are neural networks with billions of parameters trained on
    trillions of tokens across thousands of GPUs.
    """,
]

print(f"{len(DOCUMENTS)} documents loaded.")

---
## Stage 1 · Chunking

Why chunk? Embedding models have a token limit (~256–512 tokens), and shorter
chunks give more precise retrieval. Overlapping windows prevent a sentence from
being silently dropped at a boundary.

```
words  = [A, B, C, D, E, F, G]   (CHUNK_SIZE=5, CHUNK_OVERLAP=2)
chunk0 = [A, B, C, D, E]
chunk1 = [D, E, F, G]             ← D and E are shared ("overlap")
```

In [ ]:
def chunk_document(text, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    words  = text.split()
    chunks = []
    start  = 0
    while start < len(words):
        end   = start + chunk_size
        chunk = " ".join(words[start:end])
        chunks.append(chunk.strip())
        start += chunk_size - overlap
    return chunks


def build_corpus(documents):
    corpus = []
    for doc in documents:
        corpus.extend(chunk_document(doc))
    return corpus


corpus = build_corpus(DOCUMENTS)
print(f"{len(DOCUMENTS)} documents  →  {len(corpus)} chunks")
print("\nFirst 3 chunks:\n")
for i, chunk in enumerate(corpus[:3]):
    print(f"[chunk {i}]  {chunk[:140]}…\n")

---
## Stage 2 · Indexing (Embedding)

Each chunk is encoded into a **unit-normalised vector** by a sentence-transformer
model. The result is a `(N_chunks × 384)` NumPy matrix — our in-memory vector store.

Normalising to unit length means cosine similarity reduces to a plain dot product:

    cosine_similarity(a, b)  ==  dot(a_unit, b_unit)

…so retrieval is just one fast matrix multiply.

In [ ]:
def build_index(corpus, model):
    print(f"  Embedding {len(corpus)} chunks with '{EMBEDDING_MODEL}' …")
    embeddings = model.encode(corpus, show_progress_bar=True,
                              normalize_embeddings=True)
    return embeddings


print(f"Loading '{EMBEDDING_MODEL}' …")
embedding_model = SentenceTransformer(EMBEDDING_MODEL)
index = build_index(corpus, embedding_model)
print(f"\nIndex shape: {index.shape}  ({index.shape[0]} chunks × {index.shape[1]} dims)")

---
## Stage 3 · Retrieval

The query is encoded with the **same** model (query and chunks must share a vector
space). A single matrix multiply scores every chunk at once:

    scores = index @ query_vec   # shape: (N_chunks,)

The top-K chunks by score are returned as context.

In [ ]:
def retrieve(query, model, index, corpus, top_k=TOP_K):
    query_vec = model.encode(query, normalize_embeddings=True)
    scores    = index @ query_vec
    top_idxs  = np.argsort(scores)[::-1][:top_k]
    return [corpus[i] for i in top_idxs]


# Try it — change this query and re-run to explore retrieval
test_query = "How does attention work in Transformers?"
chunks = retrieve(test_query, embedding_model, index, corpus)

print(f'Query: "{test_query}"\n')
for i, chunk in enumerate(chunks, 1):
    print(f"── Chunk {i} ──")
    print(textwrap.fill(chunk, width=80))
    print()

---
## Stage 4 · Generation

The retrieved chunks are injected into a system prompt that instructs the LLM to:
1. Answer **only** from the provided context.
2. Say "I don't know" if the context is insufficient — preventing hallucination.

In [ ]:
def generate(query, context_chunks):
    context = "\n\n---\n\n".join(context_chunks)

    system_prompt = (
        "You are a helpful teaching assistant.\n"
        "Answer the student's question using ONLY the context passages below.\n"
        "If the context does not contain enough information to answer, say so clearly "
        "and do not guess.\n\n"
        f"CONTEXT:\n{context}"
    )

    response = litellm.completion(
        model      = LLM_MODEL,
        max_tokens = 512,
        api_base   = LLM_API_BASE or None,
        messages   = [
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": query},
        ],
    )
    return response.choices[0].message.content

---
## Full Pipeline

`answer()` chains all four stages together. Modify the questions below to explore.

In [ ]:
def answer(question):
    print(f"Q: {question}")
    chunks   = retrieve(question, embedding_model, index, corpus)
    response = generate(question, chunks)
    print("\nA:", textwrap.fill(response, width=70, subsequent_indent="   "))
    print()


answer("What is the attention mechanism?")
answer("What problem does RAG solve?")
answer("What is cosine similarity used for?")